# Telecom Customer Churn Analysis & Machine Learning Pipeline
This notebook provides end-to-end exploratory data analysis (EDA), feature engineering, multi-model benchmarking (Logistic Regression, Decision Tree, Random Forest, Gradient Boosting), and SHAP (SHapley Additive exPlanations) explainability for customer churn in Algerian telecom networks.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, roc_curve, classification_report

from ml.data.generate_churn import generate_churn_dataset
from ml.churn.preprocessing import TelecomChurnPreprocessor, prepare_churn_data
from ml.churn.train import train_and_evaluate_models

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

## 1. Load and Inspect Churn Dataset

In [2]:
df = generate_churn_dataset(num_samples=10000, random_seed=42)
print(f"Dataset Shape: {df.shape}")
print(f"Churn Distribution:\n{df['churn'].value_counts(normalize=True) * 100}%")
df.head()

## 2. Exploratory Data Analysis & Churn Correlations
Examining the key drivers of subscriber churn: complaints, usage drop rate, and contract type.

In [3]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Churn by Complaints
sns.barplot(data=df, x='complaints', y='churn', ax=axes[0], palette='Reds_d')
axes[0].set_title('Churn Rate by Customer Service Complaints (60d)')
axes[0].set_ylabel('Churn Probability')

# Churn by Usage Drop %
sns.boxplot(data=df, x='churn', y='usage_drop_pct', ax=axes[1], palette=['#38bdf8', '#f43f5e'])
axes[1].set_title('Monthly Usage Decline % vs Churn')
axes[1].set_xticklabels(['Stay', 'Churn'])

# Churn by Subscription Type
sns.barplot(data=df, x='subscription', y='churn', ax=axes[2], palette='Blues_d')
axes[2].set_title('Churn Rate by Subscription Model')

plt.tight_layout()
plt.show()

## 3. Train Multi-Model Benchmark & Select Champion
Training Logistic Regression, Decision Tree, Random Forest, and Gradient Boosting under cross-validation.

In [4]:
results = train_and_evaluate_models()
print(f"Selected Champion: {results['champion']}")
print(f"Selection Criterion: {results['selectionCriterion']}")
pd.DataFrame(results['models'])[['name', 'rocAuc', 'f1Score', 'precision', 'recall', 'trainTimeMs']]

## 4. SHAP Feature Importance Analysis
Inspecting the global and local TreeExplainer attributions to explain individual subscriber churn predictions.

In [5]:
import joblib
from ml.churn.evaluate import compute_shap_explanations

model = joblib.load('ml/churn/champion_model.joblib')
prep = joblib.load('ml/churn/preprocessor.joblib')

X_sample = prep.transform(df.head(200))
shap_values = compute_shap_explanations(model, X_sample, prep.feature_names)

shap_df = pd.DataFrame(shap_values, columns=prep.feature_names)
mean_abs_shap = shap_df.abs().mean().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
mean_abs_shap.plot(kind='barh', color='#0ea5e9')
plt.title('Mean Absolute SHAP Value (Global Feature Importance)')
plt.xlabel('Mean |SHAP Value|')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()